In [1]:
import matplotlib
matplotlib.use("Agg")
from scipy.integrate import odeint
from sympy import var, I
import numpy as np
import matplotlib.pyplot as plt
import math

import importlib
import sys
import os

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import set_style, colores, format_ax
set_style()

2026-03-18 21:01:05,188 - openket - INFO - openket v0.1.0 initialized successfully.


In [3]:
def P(t, rabi, delta):
    omega = np.sqrt(delta**2 + rabi**2)
    return rabi**2/omega**2 * (np.sin(omega*t/2))**2

hbar = 1
t = np.linspace(0, 10, 1000)
base = [Ket(0), Ket(1)]
rho = Operator("rho")

In [6]:
filename = "atomo"
rabi_c = 2.0
detunings = [0.0, 2.0, 4.0]

plt.close()
def sigma(i,j): return Ket(i) * Bra(j)
for i, detuning in enumerate(detunings):
    H_atomo = hbar * detuning * sigma(1,1)
    H_laser = -(hbar/2) * rabi_c * (sigma(0,1) + sigma(1,0))
    H = H_atomo + H_laser
    
    rdot = -(I/hbar)*comm(H, rho)

    build_ode(
        rho=rho,
        rdot=rdot,
        basis=base,
        filetype="Scipy",
        filename=f"func{filename}.py",
        dictname=f"dic{filename}.py"
    )
    # leer módulo dic.py
    funcname=f"func{filename}"
    if funcname in sys.modules:
        del sys.modules[funcname]
    modulo = importlib.import_module(funcname)
    modulo = importlib.reload(modulo)
    f = modulo.f
    dic = modulo.dic
    # convertir condiciones iniciales simbólicas -> numéricas
    rho0 = Ket(0)*Bra(0)
    init_conditions = init_state(rho=rho, rho0=rho0, basis=base, dic=dic)
    # solución numérica
    rho_solution = odeint(f, init_conditions, t)
    # probabilidad del estado excitado
    j = 1
    proyector = sigma(j,j)
    proyector_symb = sub_qexpr(qexpr=trace(rho * proyector, basis=base), dic=dic)
    prob = sym2num(sol=rho_solution, symbexpr=proyector_symb)
    prob = np.real(prob)

    step=10
    plt.plot(t, P(t,rabi_c,detuning), linewidth=1, color=colores[i])
    plt.scatter(t[::step], prob[::step], label={detuning}, color=colores[i], s=15, marker='s', zorder=2)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.5)
plt.axhline(1, color="gray", linestyle="--", linewidth=0.5)
plt.xlabel(r'Tiempo adimensional $\Omega_c t$')
plt.ylabel(r'Probabilidad de excitación $P_e(t)$')
ax = plt.gca()
format_ax(ax, xstep=2, ystep=2)
plt.legend()
plt.savefig(f"../figuras/fig:deltas_atomo.png")
plt.close()

for file in os.listdir("."):
    if file.endswith(".py") and file != "style.py":
        os.remove(file)